# RF driver

The chain of RF driver models, from 0D to a 2D fluid, on one driver: the BUG driver of
`illustrative_hydrogen_driver` (hydrogen at 0.3 Pa, 1 MHz), here as an infinitely long
solenoid around a plasma column, where the three models can be compared directly.

1. `RFDriver`: a global model of the plasma and a transformer model of the coil;
2. `InductiveField2D` with `DriverCoupling`: the field in r–z with the 0D plasma;
3. `Fluid2D`: the plasma as a multi-species fluid heated by that field.

The rates are placeholders (the function warns): trends, not the driver's numbers.

In [ ]:
import warnings

import numpy as np
import plasma_plots

from wellbalanced.driver import illustrative_hydrogen_driver

POWER = 25e3  # W absorbed
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    driver = illustrative_hydrogen_driver(coupling="solenoid", field_factor=1.0)

## 1. The 0D driver

The absorbed power fixes the electron density and temperature; they fix the skin layer and
the resistance the coil sees, and that the coil current and the RF power transfer
efficiency $\eta = R_p / (R_p + R_\mathrm{net})$.

In [ ]:
zero_d = driver.solve(POWER)
print(
    f"n_e = {zero_d.plasma.electron_density:.2e} m^-3, "
    f"T_e = {zero_d.plasma.electron_temperature:.2f} eV, "
    f"skin depth {zero_d.skin_depth * 1e3:.1f} mm\n"
    f"coil current {zero_d.coil_current:.0f} A, efficiency {zero_d.efficiency:.2f}"
)

## 2. The field in r–z

The column on a grid in r, fine at the axis and stretched outwards, and one cell along a
periodic z; the coil is a winding of `coil_turns` at the coil radius. `DriverCoupling`
solves the field for a plasma that depends on the power: here the 0D driver's uniform
plasma inside the driver radius (`global_model_plasma`), at the same absorbed power.

In [ ]:
from simgeometry import Rectangle

from wellbalanced.spatial import (
    CellGrid,
    DriverCoupling,
    InductiveField2D,
    Winding,
    global_model_plasma,
    stretched_edges,
)

radius = driver.radius
coil = Winding(
    Rectangle((driver.coil_radius, driver.length / 2), (0.004, 1.0)),
    sign=driver.coil_turns,
)


def column_grid(r_edges):
    """The infinitely long driver: periodic along z."""
    return CellGrid(
        r_edges,
        np.linspace(0, driver.length, 2),
        geometry="cylindrical",
        boundaries=("neumann", "periodic"),
    )


def field_on(grid):
    """The field of the coil on ``grid`` for a plasma ``profile`` inside the driver."""

    def field(profile):
        settings = profile.settings()
        density = settings.pop("electron_density")
        return InductiveField2D(
            grid=grid,
            frequency=driver.frequency,
            windings=[coil],
            electron_density=lambda r, z: np.where(r < radius, density, 0.0),
            **settings,
        )

    return field


def couple(field, plasma):
    return DriverCoupling(
        field=field, plasma=plasma, network_resistance=driver.network_resistance
    ).at_absorbed_power(POWER)


fine = column_grid(stretched_edges(0.0, 0.2, 0.0, 0.15, 5e-4, ratio=1.1))
coupled_0d = couple(field_on(fine), global_model_plasma(driver))
print(
    f"coil current {coupled_0d.coil_current:.0f} A, "
    f"efficiency {coupled_0d.efficiency:.3f}"
)

In this long solenoid the field gives the 0D transformer model's coupling, to the
discretisation error.

`DriverCoupling.at_rf_power` instead iterates to the absorbed power of a given generator
power. The field's solution is a dataset too; the power is absorbed in the skin layer at the
plasma edge.

In [ ]:
field_data = coupled_0d.solution.to_dataset().isel(z=0)
field_data = field_data.sel(r=slice(0.08, 0.2))
with plasma_plots.figure(1, 2, figsize=(10, 3.6)) as fig:
    abs(field_data.magnetic_field_z).plasma.plot.lineout(ax=fig[0])
    field_data.power_density.plasma.plot.lineout(ax=fig[1])

## 3. The fluid in its field

`Fluid2D` carries the electrons and the driver's ions (H+, H2+, H3+) on the cells of the
cells of a coarser grid (5 mm in the plasma), with the 0D driver's reactions and gas; the ions have inertia and a constant
mean free path, the potential is quasi-neutral with sheath walls. First a steady state
heated uniformly, then `fluid_plasma` couples it to the field at the same absorbed power.

In [ ]:
from plasmacoll.constants import ELEMENTARY_CHARGE as e

from wellbalanced import ELECTRON, FluidSpecies
from wellbalanced.spatial import Fluid2D, fluid_plasma

spacing = 5e-3
grid = column_grid(
    np.concatenate(
        [
            np.linspace(0, radius, round(radius / spacing) + 1),
            stretched_edges(radius, 0.2, radius, radius + 0.01, spacing, 1.2)[1:],
        ]
    )
)
in_plasma = grid.xx < radius
tg = driver.model_options["gas_temperature"]
species = [FluidSpecies(ELECTRON, 1.0)]  # its mobility follows from nu
for s in driver.species:
    if s.charge and s.name != "e":
        thermal = np.sqrt(8.0 * e * tg / (np.pi * s.mass))
        species.append(
            FluidSpecies(s, e * driver.ion_mean_free_path / (s.mass * thermal))
        )
volume = np.pi * radius**2 * driver.length
fluid = Fluid2D(
    grid,
    species,
    driver.reactions,
    region=in_plasma,
    fixed=driver.fixed,
    gas_temperature=tg,
    momentum_transfer=driver.momentum_transfer,
    ion_mean_free_path=driver.ion_mean_free_path,
    heating=np.where(in_plasma, POWER / volume, 0.0),
)
shape = np.cos(0.4 * np.pi * grid.xx / radius)
ions = {"H+": 0.25, "H2+": 0.5, "H3+": 0.25}
uniform = fluid.steady_state(
    fluid.pack({name: share * 2e18 * shape for name, share in ions.items()}, 5.0, 25.0)
)
uniform.converged

In [ ]:
field = field_on(grid)
coupled_fluid = couple(field, fluid_plasma(fluid, field, uniform, lorentz=False))
coarse_0d = couple(field, global_model_plasma(driver))
state = coupled_fluid.plasma.state
print(
    f"mean n_e = {state.mean(state.electron_density):.2e} m^-3, "
    f"coil current {coupled_fluid.coil_current:.0f} A, "
    f"efficiency {coupled_fluid.efficiency:.3f}\n"
    f"0D plasma on the same grid: coil current {coarse_0d.coil_current:.0f} A, "
    f"efficiency {coarse_0d.efficiency:.3f}"
)

The fluid's density falls towards the wall, where the skin layer is, so the coil sees a
different plasma than the 0D model's uniform one. The 5 mm cells are about a skin depth:
a quick case, not a converged one. The RF Lorentz force of the field on the electrons
(`lorentz=True`) is not applied here; the [RF driver 2D fluid](/wellbalanced/full-examples/rf-driver-fluid2d/)
example shows it, in the driver's full r–z geometry.

In [ ]:
profiles = state.to_dataset().isel(z=0).where(lambda d: d["plasma"], drop=True)
with plasma_plots.figure(1, 2, figsize=(10, 3.6)) as fig:
    profiles.density.plasma.plot.profiles(x="r", over="species", ax=fig[0])
    profiles.electron_temperature.plasma.plot.lineout(ax=fig[1])